# 09 - Generated Image Embedding (DINOv2)

## Amaç

Bu notebookta Stable Diffusion XL kullanılarak oluşturulan Türkçe ve İngilizce görseller için DINOv2 embeddingleri oluşturulmaktadır.

Bu embeddingler daha sonraki aşamada;

- Türkçe ve İngilizce istemlerin karşılaştırılması
- Gerçek ve üretilen görsellerin benzerlik analizi
- PCA
- t-SNE
- Cosine Similarity

analizlerinde kullanılacaktır.

Notebook sonunda aşağıdaki dosyalar oluşturulur:

- sdxl_tr_embeddings.npy
- sdxl_en_embeddings.npy
- sdxl_tr_image_names.csv
- sdxl_en_image_names.csv

In [12]:
import os
import numpy as np
import pandas as pd
import torch

from PIL import Image
from transformers import AutoImageProcessor, AutoModel

## DINOv2 Modelinin Yüklenmesi

Bu aşamada Facebook Research tarafından geliştirilen DINOv2 modeli yüklenmektedir.

Model tüm üretilen görseller için öznitelik (feature embedding) çıkarmak amacıyla kullanılacaktır.

In [13]:
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")

model = AutoModel.from_pretrained("facebook/dinov2-base")

model.eval()

print("DINOv2 modeli başarıyla yüklendi.")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 modeli başarıyla yüklendi.


## Görsel Klasörlerinin Okunması

Bu aşamada Türkçe ve İngilizce istemlerle oluşturulan Stable Diffusion XL görselleri okunmaktadır.

In [14]:
TR_FOLDER = "../data/CHRD_Generated/SDXL_TR"

EN_FOLDER = "../data/CHRD_Generated/SDXL_EN"

tr_images = sorted(os.listdir(TR_FOLDER))

en_images = sorted(os.listdir(EN_FOLDER))

print("TR :", len(tr_images))

print("EN :", len(en_images))

TR : 96
EN : 96


# Görsellerin DINOv2 Modeline Hazırlanması

Bu bölümde üretilen SDXL görselleri DINOv2 modeline uygun hale getirilmektedir.

Her görüntü:

- diskten okunur,
- RGB formatına dönüştürülür,
- yeniden boyutlandırılır,
- Tensor formatına çevrilir,
- normalize edilir.

Bu işlem sonucunda görüntüler embedding üretimi için hazır hale gelir.

In [15]:
def load_image(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    return inputs

# Türkçe Görseller İçin Embedding Üretimi

Bu bölümde Türkçe istemlerle oluşturulan SDXL görselleri DINOv2 modeli ile işlenmektedir.

In [16]:
tr_embeddings = []

tr_image_names = []

for file_name in tr_images:

    image_path = os.path.join(TR_FOLDER, file_name)

    inputs = load_image(image_path)

    with torch.no_grad():

        outputs = model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

    tr_embeddings.append(embedding)

    tr_image_names.append(file_name)

tr_embeddings = np.array(tr_embeddings)

print("TR Embedding Shape :", tr_embeddings.shape)

TR Embedding Shape : (96, 768)


# İngilizce Görseller İçin Embedding Üretimi

Bu bölümde İngilizce istemlerle oluşturulan SDXL görselleri aynı DINOv2 modeli kullanılarak işlenmektedir.

In [17]:
en_embeddings = []

en_image_names = []

for file_name in en_images:

    image_path = os.path.join(EN_FOLDER, file_name)

    inputs = load_image(image_path)

    with torch.no_grad():

        outputs = model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

    en_embeddings.append(embedding)

    en_image_names.append(file_name)

en_embeddings = np.array(en_embeddings)

print("EN Embedding Shape :", en_embeddings.shape)

EN Embedding Shape : (96, 768)


# Embedding Boyutlarının Kontrolü

Bu bölümde oluşturulan embeddinglerin boyutları kontrol edilmektedir.

In [18]:
print("Türkçe :", tr_embeddings.shape)

print("İngilizce :", en_embeddings.shape)

Türkçe : (96, 768)
İngilizce : (96, 768)


# Embeddinglerin Kaydedilmesi

Bu bölümde oluşturulan embeddingler ve dosya isimleri sonraki notebooklarda kullanılmak üzere kaydedilmektedir.

In [19]:
os.makedirs("../embeddings", exist_ok=True)

In [20]:
np.save(
    "../embeddings/sdxl_tr_embeddings.npy",
    tr_embeddings
)

np.save(
    "../embeddings/sdxl_en_embeddings.npy",
    en_embeddings
)

In [21]:
pd.DataFrame({
    "file_name": tr_image_names
}).to_csv(
    "../embeddings/sdxl_tr_image_names.csv",
    index=False
)

pd.DataFrame({
    "file_name": en_image_names
}).to_csv(
    "../embeddings/sdxl_en_image_names.csv",
    index=False
)

In [22]:
print("Embedding dosyaları başarıyla kaydedildi.")

print()

print(sorted(os.listdir("../embeddings")))

Embedding dosyaları başarıyla kaydedildi.

['chrd_real_embeddings.npy', 'chrd_real_image_names.csv', 'sdxl_en_embeddings.npy', 'sdxl_en_image_names.csv', 'sdxl_tr_embeddings.npy', 'sdxl_tr_image_names.csv']
